In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torch.utils.data import random_split
from tqdm.notebook import tqdm

In [ ]:
# Define transformations for the images
transform = transforms.Compose([
    transforms.ToTensor(),  # Convert images to PyTorch tensors
    transforms.Normalize((0.1307,), (0.3081,))  # Normalize with MNIST mean and std
])

In [ ]:
# Load the training dataset
train_dataset = torchvision.datasets.MNIST(
    root='./data',  # Directory to store the dataset
    train=True,  # Load training data
    download=True,  # Download if not present
    transform=transform  # Apply transformations
)

# Load the test dataset
test_dataset = torchvision.datasets.MNIST(
    root='./data',  # Directory to store the dataset
    train=False,  # Load test data
    download=True,  # Download if not present
    transform=transform  # Apply transformations
)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
# Get a batch of training data
images, labels = next(iter(train_loader))

# Print the batch size
print(f"Batch size: {images.size(0)}")

# Print the shape of each image in the batch
print(f"Image shape: {images.shape[1:]}")

Batch size: 64
Image shape: torch.Size([1, 28, 28])


In [ ]:
# Define the sizes for training and validation sets
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size

# Split the training dataset
train_subset, val_subset = random_split(train_dataset, [train_size, val_size])

In [ ]:
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

# Get a batch from the training loader to access labels
images, labels = next(iter(train_loader))

# Now y_train is not empty, so you can access its elements
print(class_names[labels[0].item()])

Bag


In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=30):
    for epoch in tqdm(range(num_epochs), desc="Training Epochs"):
        model.train()
        running_loss = 0.0
        train_correct = 0
        for images, labels in tqdm(train_loader, desc="Training Batch", leave=False):
            labels = labels.long()
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

            # Calcular acertos
            _, predicted = torch.max(outputs.data, 1)
            train_correct += (predicted == labels).sum().item()

        # Validação
        model.eval()
        val_loss = 0.0
        val_correct = 0
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc="Validation Batch", leave=False):
                labels = labels.long()
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_correct += (predicted == labels).sum().item()

        # Cálculo das métricas finais
        train_acc = train_correct / len(train_loader.dataset)
        val_acc = val_correct / len(val_loader.dataset)
        avg_train_loss = running_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)

        print(f"Epoch [{epoch+1}/{num_epochs}] "
              f"Train Acc: {train_acc:.4f} | Train Loss: {avg_train_loss:.4f} | "
              f"Val Acc: {val_acc:.4f} | Val Loss: {avg_val_loss:.4f}")

In [ ]:
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28*28, 300),
    nn.ReLU(),
    nn.Linear(300, 100),
    nn.ReLU(),
    nn.Linear(100, 10),
    nn.Softmax(dim=1)
)

In [ ]:
for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Total Params: {param.numel()}")

Layer: 1.weight | Size: torch.Size([300, 784]) | Total Params: 235200
Layer: 1.bias | Size: torch.Size([300]) | Total Params: 300
Layer: 3.weight | Size: torch.Size([100, 300]) | Total Params: 30000
Layer: 3.bias | Size: torch.Size([100]) | Total Params: 100
Layer: 5.weight | Size: torch.Size([10, 100]) | Total Params: 1000
Layer: 5.bias | Size: torch.Size([10]) | Total Params: 10


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)

In [ ]:
train_model(model, train_loader, test_loader, criterion, optimizer)

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

In [ ]:
housing = fetch_california_housing()

X_train_full, X_test, y_train_full, y_test = train_test_split(
    housing.data, housing.target, random_state=42)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, random_state=42
)

scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_val_scaled = scaler_X.transform(X_val)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1))
y_val_scaled = scaler_y.transform(y_val.reshape(-1, 1))

In [ ]:
def train_model_regressor(model, X_train, y_train, X_val, y_val, criterion, optimizer, num_epochs=20, device=None):
    # Detectar device automaticamente
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Converter para tensores e mover pro device
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1).to(device)
    X_val_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_val_tensor = torch.tensor(y_val, dtype=torch.float32).view(-1, 1).to(device)
    train_losses, val_losses = [], []

    for epoch in tqdm(range(num_epochs), desc="Training Epochs"):
        model.train()
        optimizer.zero_grad()

        # Forward + Backward

        outputs = model(X_train_tensor)
        loss = criterion(outputs, y_train_tensor)
        loss.backward()
        optimizer.step()

        # Avaliação
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val_tensor)
            val_loss = criterion(val_outputs, y_val_tensor)

            # Calcular R² (coeficiente de determinação)
            ss_res = torch.sum((y_val_tensor - val_outputs) ** 2)
            ss_tot = torch.sum((y_val_tensor - torch.mean(y_val_tensor)) ** 2)
            r2_score = 1 - ss_res / ss_tot


        train_losses.append(loss.item())
        val_losses.append(val_loss.item())


        print(f"Epoch [{epoch+1}/{num_epochs}] "
              f"Train Loss: {loss.item():.4f} | "
              f"Val Loss: {val_loss.item():.4f} | "
              f"R²: {r2_score.item():.4f}")

    plt.plot(train_losses, label='Train')
    plt.plot(val_losses, label='Val')
    plt.legend()
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Evolução da perda')
    plt.show()

In [ ]:
model = nn.Sequential(
    nn.Linear(8, 30),
    nn.ReLU(),
    nn.Linear(30, 1)
)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
train_model_regressor(model, X_train_scaled, y_train_scaled, X_val_scaled, y_val_scaled, criterion, optimizer)

In [ ]:
class ModelMultOutput(nn.Module):
    def __init__(self):
        super(ModelMultOutput, self).__init__()
        self.hidden1 = nn.Linear(8, 30)
        self.hidden2 = nn.Linear(30, 30)
        self.output1 = nn.Linear(30, 1)
        self.output2 = nn.Linear(30, 1)

    def forward(self, x):
        x = F.relu(self.hidden1(x))
        x = F.relu(self.hidden2(x))
        out1 = self.output1(x)
        out2 = self.output2(x)
        return out1, out2

In [ ]:
model = ModelMultOutput()

x = torch.randn(4, 8)  # batch de 4
out1, out2 = model(x)
print(out1.shape)  # [4, 1]
print(out2.shape)

torch.Size([4, 1])
torch.Size([4, 1])


In [ ]:
class ModelMultInput(nn.Module):
    def __init__(self):
        super(ModelMultInput, self).__init__()
        self.hidden1 = nn.Linear(8, 30)
        self.hidden2 = nn.Linear(30, 30)
        self.output = nn.Linear(60, 1)

    def forward(self, a, b):
        hidden_1_a = F.relu(self.hidden1(a))
        hidden_2_a = F.relu(self.hidden2(hidden_1_a))
        concat = torch.cat([hidden_2_a, b], dim=1)
        x = self.output(concat)
        return x

In [ ]:
model = ModelMultInput()
a = torch.randn(30, 8)
b = torch.randn(30, 30)
output = model(a, b)
print(output.shape)

torch.Size([30, 1])


In [ ]:
class WideAndDeep(nn.Module):
    def __init__(self, units=30):
        super().__init__()
        # deep path processa input_b (30 features)
        self.hidden1 = nn.Linear(30, units)
        self.hidden2 = nn.Linear(units, units)
        # main output recebe concat de input_a (8) + hidden2 (units)
        self.main_output = nn.Linear(8 + units, 1)
        # aux output só depende do deep path
        self.aux_output = nn.Linear(units, 1)

    def forward(self, inputs):
        input_a, input_b = inputs
        hidden1 = F.relu(self.hidden1(input_b))
        hidden2 = F.relu(self.hidden2(hidden1))
        concat = torch.cat([input_a, hidden2], dim=1)
        main_output = self.main_output(concat)
        aux_output = self.aux_output(hidden2)
        return main_output, aux_output

In [ ]:
model = WideAndDeep()
a = torch.randn(30, 8)
b = torch.randn(30, 30)
main_output, aux_output = model((a, b))
print(main_output.shape)
print(aux_output.shape)

torch.Size([30, 1])
torch.Size([30, 1])


In [ ]:
torch.save(model.state_dict(), 'model.pth')

In [ ]:
model_load = WideAndDeep()
model_load.load_state_dict(torch.load('model.pth'))

<All keys matched successfully>